# 05 — Transfer Learning ACC Arena → Salt&Tar (Advanced, 3 pts)
Reuse the NN trained on **ACC Arena** and **fine-tune** it on a *limited* **Salt&Tar** training set, comparing
against the **same network trained from scratch** on that same limited set.

Since the value of transfer learning depends on **how scarce** target-domain data is, we sweep the size of the
limited train set (number of Salt&Tar users) and evaluate both approaches on one **fixed test set**: the classic
TL curve. Feature schema is identical to notebook 02 (fixed numeric + 6 traffic one-hot + `BEST_X` neighbours
in the `BEST_ENC` encoding), so the ACC-trained weights transfer directly.

In [ ]:
# === Project config — Team 8: Throughput Prediction in a Dense 5G deployment (with Transfer Learning) ===
RANDOM_SEED      = 42
RESAMPLE_SECONDS = 120           # time granularity: THE dataset-size knob (professor's guidance: coarsen the
                                 # time grid rather than subsample users). Raw cadence is ~3s with frequent 4s
                                 # steps, duplicate stamps and gaps up to 7s (ACC) / ~1s (Salt&Tar); each output row aggregates every raw sample in a fixed-width window.
N_USERS          = None          # ACC Arena users. None = ALL (~12k), so the X-closest neighbourhoods are the
                                 # real ones; an int (e.g. 1500, seeded random sample) biases the neighbourhoods
                                 # (X closest searched within the sample) and is only for quick debug runs.
N_USERS_SALT     = 3000          # Salt&Tar users: ALL of them (only ~1/3 are ever active; activity is id-biased)
X_VALUES         = [3, 5, 10]    # number of closest users to experiment with. X=1 excluded by design:
                                 # heavy co-location makes a single arbitrary neighbour uninformative.
                                 # X=0 (no neighbour features) IS produced and trained: it is the baseline
                                 # that quantifies what the closest-users features actually add.
ENCODINGS        = ["pos", "agg"]# neighbour-feature encodings: "pos" = ordered per-neighbour columns
                                 # (nb0_*, nb1_*, ...), "agg" = order-invariant aggregates over the X
                                 # neighbours (sum/mean/count) — rationale in notebook 02.
BEST_X           = 3             # X used for the transfer-learning experiment (pick the best from notebook 04)
BEST_ENC         = "pos"         # encoding used for the transfer-learning experiment (pick from notebook 04)
OUTLIER_PCT      = None          # optional train-only sensitivity analysis; primary results keep the full distribution.
                                 # EDA (notebook 01): the ~1% extreme samples carry ~2/3 of the total variance.
ACTIVE_ONLY      = False          # primary task covers every user; True is an optional active-only sensitivity run
MIN_TRAFFIC_TYPE = 2             # keep traffic_type >= this (2=const_rate, 3=video, 4=gaming, 5=http)

# --- data location ---
DRIVE_FOLDER_ID  = "1s1YCWyhN_Fv5sQraTVs4Rga-ATiKPRgo"   # shared Drive folder containing the zip
ZIP_NAME         = "L5GHDD_Dataset.zip"
DATA_ROOT        = "data/raw"     # dataset folders live here after unzip (local default)
PROCESSED_DIR    = "data/processed"
RESULTS_DIR      = "results"

import os, numpy as np, random
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/figures", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/models", exist_ok=True)

In [ ]:
# === Colab: mount Drive and make outputs PERSIST across notebooks (no-op locally) ===
def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

if in_colab():
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR   = "/content/drive/MyDrive/5g_throughput_team8"   # persistent project folder on your Drive
    PROCESSED_DIR = f"{PROJECT_DIR}/processed"                     # 02 writes here, 03/04/05 read from here
    RESULTS_DIR   = f"{PROJECT_DIR}/results"                       # models, metrics.csv, figures
    print("Outputs persist in:", PROJECT_DIR)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/figures", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/models", exist_ok=True)

In [ ]:
# === Colab: locate and unzip the dataset (no-op locally) ===
if in_colab():
    import glob, zipfile, subprocess
    DATA_ROOT = "/content/L5GHDD_Dataset"
    if not os.path.isdir(DATA_ROOT):
        cands = glob.glob(f"/content/drive/MyDrive/**/{ZIP_NAME}", recursive=True)
        if not cands:                                   # fallback: download the shared folder
            subprocess.run(["pip", "-q", "install", "gdown"], check=False)
            import gdown
            gdown.download_folder(id=DRIVE_FOLDER_ID, output="/content/_dl", quiet=False, use_cookies=False)
            cands = glob.glob(f"/content/_dl/**/{ZIP_NAME}", recursive=True)
        assert cands, f"{ZIP_NAME} not found. Put it in your Drive or share the folder."
        print("Unzipping", cands[0], "...")
        with zipfile.ZipFile(cands[0]) as z:
            z.extractall(DATA_ROOT)
print("DATA_ROOT =", DATA_ROOT)

In [ ]:
# === Data loading helpers (raw wide format -> uniform tidy windows) ===
import glob, re, math
import pandas as pd
from scipy.spatial import cKDTree

def find_venue_dir(data_root, venue_key):
    """venue_key in {'acc','salt'}; robust to the zip's internal layout."""
    pat = {"acc": "*ACC*Arena*", "salt": "*Salt*Tar*"}[venue_key]
    hits = [os.path.join(dp, d) for dp, dn, _ in os.walk(data_root)
            for d in dn if glob.fnmatch.fnmatch(d, pat)]
    hits = sorted(set(hits), key=len)
    assert hits, f"venue '{venue_key}' not found under {data_root}"
    return hits[0]

def file_id_range(path):
    m = re.search(r'_(\d+)_(\d+)\.csv$', path)
    return int(m.group(1)), int(m.group(2))

def metric_files(venue_dir, subdir_glob, file_glob, user_ids=None):
    subs = glob.glob(os.path.join(venue_dir, subdir_glob))
    assert subs, f"no subdir matching {subdir_glob} in {venue_dir}"
    files = sorted(glob.glob(os.path.join(subs[0], file_glob)), key=lambda p: file_id_range(p)[0])
    if user_ids is not None:
        def overlaps(path):
            first, last = file_id_range(path)
            return any(first <= user <= last for user in user_ids)
        files = [path for path in files if overlaps(path)]
    assert files, f"no files matching {file_glob} in {subdir_glob}"
    return files

def all_user_ids(venue_dir):
    ids = []
    for path in metric_files(venue_dir, "*Throughput*", "*.csv"):
        first, last = file_id_range(path)
        ids.extend(range(first, last + 1))
    return np.asarray(sorted(ids))

def _time_origin(venue_dir):
    """Common origin shared by both raw timelines in a venue."""
    starts = []
    for path in glob.glob(os.path.join(venue_dir, "**", "*.csv"), recursive=True):
        starts.append(float(pd.read_csv(path, usecols=[0], nrows=1).iloc[0, 0]))
    return min(starts)

def _window_index(times, origin, seconds):
    return origin + np.floor((np.asarray(times, dtype=float) - origin) / seconds) * seconds

def load_metric(files, value_name, origin, seconds, user_ids, how="mean"):
    """Aggregate every raw observation into a deterministic, uniform time window."""
    out = []
    for path in files:
        header = list(pd.read_csv(path, nrows=0).columns)
        column_to_user = {c: int(re.search(r'(\d+)', c).group(1)) for c in header[1:]}
        usecols = [header[0]] + [c for c in header[1:] if column_to_user[c] in user_ids]
        wide = pd.read_csv(path, usecols=usecols).rename(columns={header[0]: "raw_time"})
        wide["time"] = _window_index(wide.pop("raw_time"), origin, seconds)
        values = wide.groupby("time", sort=True)
        wide = (values.mean() if how == "mean" else values.last()).astype("float32")
        wide = wide.rename(columns=column_to_user)
        out.append(wide.reset_index().melt(id_vars="time", var_name="user_id", value_name=value_name))
    return pd.concat(out, ignore_index=True)

def load_positions(files, origin, seconds, user_ids):
    """Aggregate positions per window and convert latitude/longitude to local metres."""
    frames = []
    for path in files:
        first = pd.read_csv(path, nrows=1).values.astype(float)
        all_ids = first[0, 1::5].astype(int)
        blocks = [k for k, user in enumerate(all_ids) if user in user_ids]
        if not blocks:
            continue
        usecols = [0] + [1 + 5 * k + j for k in blocks for j in range(5)]
        raw = pd.read_csv(path, usecols=usecols).values.astype(float)
        ids = raw[0, 1::5].astype(int)
        bins = _window_index(raw[:, 0], origin, seconds)
        pieces = []
        for name, values, how in [
            ("lat", raw[:, 2::5], "mean"), ("lon", raw[:, 3::5], "mean"),
            ("z", raw[:, 4::5], "mean"), ("traffic_type", raw[:, 5::5], "last")]:
            wide = pd.DataFrame(values, index=bins, columns=ids)
            wide = wide.groupby(level=0, sort=True)
            wide = wide.mean() if how == "mean" else wide.last()
            wide.index.name = "time"
            pieces.append(wide.reset_index().melt(id_vars="time", var_name="user_id", value_name=name))
        frame = pieces[0]
        for piece in pieces[1:]:
            frame = frame.merge(piece, on=["time", "user_id"], validate="one_to_one")
        frames.append(frame)
    pos = pd.concat(frames, ignore_index=True)
    lat0, lon0 = pos.lat.mean(), pos.lon.mean()
    radius_m = 6_371_000.0
    pos["x"] = radius_m * np.radians(pos.lon - lon0) * math.cos(math.radians(lat0))
    pos["y"] = radius_m * np.radians(pos.lat - lat0)
    return pos[["time", "user_id", "x", "y", "z", "traffic_type"]]

def assemble(venue_key, n_users, resample_seconds, random_users=False):
    venue = find_venue_dir(DATA_ROOT, venue_key)
    population = all_user_ids(venue)
    if n_users is None:
        user_ids = set(map(int, population))
        print(f"{venue_key}: using ALL {len(user_ids)} users")
    elif random_users:
        rng = np.random.default_rng(RANDOM_SEED)
        user_ids = set(map(int, rng.choice(population, size=min(n_users, len(population)), replace=False)))
        print(f"{venue_key}: sampled {len(user_ids)} random users out of {len(population)}")
    else:
        user_ids = set(map(int, population[:n_users]))
    origin = _time_origin(venue)
    mf = lambda sub, pattern: metric_files(venue, sub, pattern, user_ids)
    parts = [
        load_metric(mf("*Throughput*", "*.csv"), "throughput", origin, resample_seconds, user_ids),
        load_metric(mf("*BLER*", "*.csv"), "bler", origin, resample_seconds, user_ids),
        load_metric(mf("*PRB*", "*.csv"), "prb", origin, resample_seconds, user_ids),
        load_metric(mf("*RU_Association*", "*.csv"), "ru_id", origin, resample_seconds, user_ids, how="last"),
        load_metric(mf("*SINR*", "SINRDL_*.csv"), "sinr_dl", origin, resample_seconds, user_ids),
        load_metric(mf("*SINR*", "SINRUL_*.csv"), "sinr_ul", origin, resample_seconds, user_ids),
        load_positions(mf("*Positions*", "*.csv"), origin, resample_seconds, user_ids),
    ]
    frame = parts[0]
    for part in parts[1:]:
        frame = frame.merge(part, on=["time", "user_id"], how="inner", validate="one_to_one")
    frame = frame.dropna().reset_index(drop=True)
    frame["user_id"] = frame.user_id.astype(int)
    frame["traffic_type"] = frame.traffic_type.round().astype(int)
    frame["ru_id"] = frame.ru_id.round().astype(int)
    assert not frame.duplicated(["time", "user_id"]).any()
    return frame


In [ ]:
# === Closest-users feature engineering (Team-8 specific) ===
# The target variable is deliberately excluded: neighbour throughput would leak labels across user splits.
NEIGHBOR_FEATS = ["sinr_dl", "sinr_ul", "prb", "bler", "traffic_type"]

def add_closest_user_features(df, x_max):
    cols = []
    for k in range(x_max):
        cols += [f"nb{k}_dist"] + [f"nb{k}_{c}" for c in NEIGHBOR_FEATS]
    out = np.full((len(df), len(cols)), np.nan, dtype="float32")
    feat = df[NEIGHBOR_FEATS].values
    pos = df[["x", "y", "z"]].values
    for _, idx in df.groupby("time", sort=False).groups.items():
        idx = np.asarray(idx)
        n = len(idx)
        if n <= 1:
            continue
        k = min(x_max + 1, n)
        dist, neighbours = cKDTree(pos[idx]).query(pos[idx], k=k)
        if k == 1:
            dist, neighbours = dist[:, None], neighbours[:, None]
        rows = np.arange(n)[:, None]
        order = np.argsort(neighbours == rows, axis=1, kind="stable")
        take = min(x_max, k - 1)
        r = np.repeat(np.arange(n), take)
        c = order[:, :take].ravel()
        block = np.empty((n, take, 1 + len(NEIGHBOR_FEATS)), dtype="float32")
        block[:, :, 0] = dist[r, c].reshape(n, take)
        block[:, :, 1:] = feat[idx[neighbours[r, c]]].reshape(n, take, -1)
        out[idx, :take * (1 + len(NEIGHBOR_FEATS))] = block.reshape(n, -1)
    return pd.concat([df, pd.DataFrame(out, columns=cols, index=df.index)], axis=1)

def neighbor_cols(x):
    return [name for k in range(x)
            for name in [f"nb{k}_dist"] + [f"nb{k}_{feature}" for feature in NEIGHBOR_FEATS]]

# === Order-invariant neighbour aggregates (encoding "agg") ===
AGG_FEATS = ["nb_prb_sum", "nb_sinr_dl_mean", "nb_sinr_ul_mean", "nb_bler_mean",
             "nb_active_count", "nb_distance_mean"]

def aggregate_neighbor_features(frame, x):
    def block(feature):
        return frame[[f"nb{k}_{feature}" for k in range(x)]]
    prb = block("prb")
    traffic = block("traffic_type")
    return pd.DataFrame({
        "nb_prb_sum": prb.sum(axis=1, min_count=1),
        "nb_sinr_dl_mean": block("sinr_dl").mean(axis=1),
        "nb_sinr_ul_mean": block("sinr_ul").mean(axis=1),
        "nb_bler_mean": block("bler").mean(axis=1),
        "nb_active_count": (traffic >= MIN_TRAFFIC_TYPE).sum(axis=1).astype(float),
        "nb_distance_mean": block("dist").mean(axis=1),
    }, index=frame.index)


In [ ]:
# === Evaluation metrics ===
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
def evaluate(y_true, y_pred):
    mse = float(mean_squared_error(y_true, y_pred))
    return {"MSE": mse, "RMSE": float(np.sqrt(mse)),
            "MAE": float(mean_absolute_error(y_true, y_pred)),
            "R2": float(r2_score(y_true, y_pred))}

## Build the Salt&Tar dataset (limited) with the same feature schema

In [ ]:
import json, joblib, numpy as np, pandas as pd
df = assemble("salt", n_users=N_USERS_SALT, resample_seconds=RESAMPLE_SECONDS)
if BEST_X > 0:
    df = add_closest_user_features(df, x_max=BEST_X)
if ACTIVE_ONLY:
    df = df[df.traffic_type >= MIN_TRAFFIC_TYPE].reset_index(drop=True)
print("Salt&Tar tidy:", df.shape, "| traffic types:", sorted(df.traffic_type.unique()))

BASE_NUM = ["bler", "prb", "sinr_dl", "sinr_ul", "x", "y", "z"]
TRAFFIC_CLASSES = [0, 1, 2, 3, 4, 5]
SFX = "_agg" if BEST_ENC == "agg" else ""
with open(f"{PROCESSED_DIR}/acc_X{BEST_X}{SFX}_cols.json") as handle:
    acc_cols = json.load(handle)
preprocessor = joblib.load(f"{PROCESSED_DIR}/acc_X{BEST_X}{SFX}_preprocessor.pkl")
scaler, train_medians = preprocessor["scaler"], preprocessor["medians"]

def build_like_acc(frame):
    if BEST_ENC == "agg" and BEST_X > 0:
        frame = pd.concat([frame, aggregate_neighbor_features(frame, BEST_X)], axis=1)
        num_cols = BASE_NUM + AGG_FEATS
    else:
        num_cols = BASE_NUM + neighbor_cols(BEST_X)
    numeric = frame[num_cols].astype("float32").fillna(train_medians)
    numeric = pd.DataFrame(scaler.transform(numeric), columns=num_cols, index=frame.index)
    traffic = pd.DataFrame({f"traffic_{c}": (frame.traffic_type == c).astype(float)
                            for c in TRAFFIC_CLASSES}, index=frame.index)
    matrix = pd.concat([numeric, traffic], axis=1).reindex(columns=acc_cols, fill_value=0.0)
    return matrix.to_numpy("float32")


## Fixed test set + growing "limited" train sets
Half of the Salt&Tar users form a **fixed test set** used for every run. The remaining users form the training
pool; each sweep step takes the first `n` users of the pool. The outlier threshold is computed once on the
training pool (never on test) so the evaluation scope is identical across all sizes.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
users = df.user_id.unique(); rng.shuffle(users)
n_test = len(users) // 2
test_users = set(users[:n_test])
pool_users = list(users[n_test:])
TRAIN_SIZES = [5, 10, 25, 50, len(pool_users)]

test = df[df.user_id.isin(test_users)]
pool = df[df.user_id.isin(pool_users)]
# The fixed test set and validation pool always retain the full target distribution.
Xte, yte = build_like_acc(test), test.throughput.to_numpy("float32")
print("fixed test rows:", len(Xte), "| train pool users:", len(pool_users), "| sweep:", TRAIN_SIZES)


## Fine-tuned vs from-scratch at every train size
Same architecture and stopping rule for both; the only differences are the starting weights (ACC-trained vs
random) and the fine-tuning learning rate (lower, to protect the pretrained weights; no layer is frozen
because the target venue's throughput scale differs from the source's).

In [ ]:
import time
from tensorflow import keras

def make_finetune():
    model = keras.models.load_model(f"{RESULTS_DIR}/models/nn_X{BEST_X}{SFX}.keras")
    model.compile(optimizer=keras.optimizers.Adam(3e-4), loss="mse", metrics=["mae"])
    return model

def make_scratch():
    source = keras.models.load_model(f"{RESULTS_DIR}/models/nn_X{BEST_X}{SFX}.keras")
    model = keras.models.clone_model(source)
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse", metrics=["mae"])
    return model

rows = []
for n_users in TRAIN_SIZES:
    chosen = np.asarray(pool_users[:n_users])
    # Validation users are disjoint from training users; no row-level validation_split leakage.
    n_val = max(1, int(round(0.2 * len(chosen)))) if len(chosen) > 1 else 0
    val_users = set(chosen[-n_val:]) if n_val else set()
    train_users = set(chosen[:-n_val]) if n_val else set(chosen)
    train = pool[pool.user_id.isin(train_users)]
    val = pool[pool.user_id.isin(val_users)]
    if OUTLIER_PCT is not None:
        threshold = float(np.percentile(train.throughput, OUTLIER_PCT))
        train = train[train.throughput <= threshold]
    Xtr, ytr = build_like_acc(train), train.throughput.to_numpy("float32")
    validation = (build_like_acc(val), val.throughput.to_numpy("float32")) if len(val) else None
    for setting, factory in [("fine-tuned (TL)", make_finetune), ("from scratch", make_scratch)]:
        keras.utils.set_random_seed(RANDOM_SEED)
        model = factory()
        stop = keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True)
        start = time.time()
        model.fit(Xtr, ytr, validation_data=validation, epochs=60, batch_size=256,
                  callbacks=[stop], verbose=0)
        result = evaluate(yte, model.predict(Xte, verbose=0).ravel())
        result.update(setting=setting, train_users=len(train_users), requested_users=n_users,
                      train_rows=len(Xtr), train_s=round(time.time() - start, 1))
        rows.append(result)
        print(f"users={n_users:>4} | {setting:16s} R2={result['R2']:.3f} MAE={result['MAE']:.3f}")

tl = pd.DataFrame(rows)
tl.to_csv(f"{RESULTS_DIR}/transfer_learning.csv", index=False)
tl


## Compare — where does transfer learning pay off?

In [ ]:
import matplotlib.pyplot as plt
sizes = sorted(tl.train_users.unique())
xs = list(range(len(sizes)))
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for setting, color in [("fine-tuned (TL)", "#2a9d8f"), ("from scratch", "#e76f51")]:
    d = tl[tl.setting == setting].set_index("train_users").loc[sizes]
    ax[0].plot(xs, d.R2,  marker="o", color=color, label=setting)
    ax[1].plot(xs, d.MAE, marker="o", color=color, label=setting)
for a, name in zip(ax, ["R2 (fixed test set)", "MAE (fixed test set, Mbps)"]):
    a.set_xticks(xs); a.set_xticklabels(sizes)
    a.set_xlabel("Salt&Tar users in the limited train set"); a.set_title(name)
    a.legend(); a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/figures/05_transfer_learning.png", dpi=120); plt.show()

**Expected reading.** With very few target-domain users the from-scratch network cannot learn a good mapping and
the ACC-pretrained one dominates; as the limited set grows the two curves converge (and from-scratch may even
edge ahead, since the pretrained weights are tuned to the source venue's throughput scale). The value of transfer learning is the gap in the
low-data regime — report the smallest train size at which each approach becomes usable.